# Cluster Labeling

The purpose of this notebook is to label our clusters with the help of hand-labeled samples (`replay_labels.csv`).
Every labeled replay in a cluster gives votes, and the votes decide the label of the cluster.
Each clustering method has its own execute cell, so they can be run one at a time.

**Index**

- libraries
- settings
- import data
- voting logic
- execute
    - 03b - movement & camera
    - 04b - mcplayhd concept

## libraries

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

## settings

In [ ]:
DATA_DIR = Path("../data")
EXPORT_DIR = DATA_DIR / "notebooks_exports"

REPLAY_LABELS_PATH = DATA_DIR / "replay_labels.csv"

# method -> clustering result csv written by 03_data-analysis
CLUSTER_EXPORT_PATHS = {
    "movement_camera": EXPORT_DIR / "labeling_cluster_export_movement_camera.csv",
    "mcplayhd_concept": EXPORT_DIR / "labeling_cluster_export_mcplayhd_concept.csv",
}

# method -> where its cluster labels get written
CLUSTER_LABELS_PATHS = {
    "movement_camera": EXPORT_DIR / "cluster_labels_movement_camera.csv",
    "mcplayhd_concept": EXPORT_DIR / "cluster_labels_mcplayhd_concept.csv",
}

PLAYSTYLE_SEPARATOR = "_"  # splits a playstyle into its parts, each part is one vote
LABEL_JOINER = "+"         # joins the winning parts into the cluster label
MIN_PART_VOTES = 3         # a part needs at least this many votes to make it into the label
MIN_LABELED_MEMBERS = 3    # clusters with fewer labeled members get TOO_SMALL_LABEL
UNCLEAR_LABEL = "unclear"
TOO_SMALL_LABEL = "too_small"

## import data

In [ ]:
df_replay_labels = pd.read_csv(REPLAY_LABELS_PATH)
print(f"{len(df_replay_labels)} labeled replays")

## voting logic
- join the replay labels onto a cluster export by `idAnonymous`

- outliers don't vote
- split each playstyle on `_`, every part is one vote
- parts with at least `MIN_PART_VOTES` votes are joined alphabetically into the label, otherwise `unclear`
- clusters with fewer than `MIN_LABELED_MEMBERS` labeled members are `too_small`
- suspicious = median of the members' suspicious values

In [ ]:
# TODO: your voting helpers go here

In [ ]:
def vote_cluster(df_export):
    """takes the voters of one cluster and returns its label columns"""
    return {}

def label_clusters(df_export):
    """takes one cluster export and returns one row per (mode, cluster) with its label"""
    df_voters = df_export.merge(df_replay_labels, on="idAnonymous", how="inner")
    df_voters = df_voters[~df_voters["is_outlier"]]

    rows = []
    for (mode, cluster), df_members in df_export.groupby(["mode", "cluster"]):
        df_cluster_voters = df_voters[(df_voters["mode"] == mode) & (df_voters["cluster"] == cluster)]
        row = {
            "mode": mode,
            "cluster": cluster,
            "cluster_size": len(df_members),
            "voters": len(df_cluster_voters),
        }
        row.update(vote_cluster(df_cluster_voters))
        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
# temp
METHOD = "mcplayhd_concept"

df_export_mcplayhd_concept = pd.read_csv(CLUSTER_EXPORT_PATHS[METHOD])
df_cluster_labels_mcplayhd_concept = label_clusters(df_export_mcplayhd_concept)

## execute

One cell per clustering method. Each cell loads its export, votes and writes its own labels csv.

### 03b - movement & camera

In [ ]:
METHOD = "movement_camera"

df_export_movement_camera = pd.read_csv(CLUSTER_EXPORT_PATHS[METHOD])
df_cluster_labels_movement_camera = label_clusters(df_export_movement_camera)
# df_cluster_labels_movement_camera.to_csv(CLUSTER_LABELS_PATHS[METHOD], index=False)

### 04b - mcplayhd concept

In [ ]:
METHOD = "mcplayhd_concept"

df_export_mcplayhd_concept = pd.read_csv(CLUSTER_EXPORT_PATHS[METHOD])
df_cluster_labels_mcplayhd_concept = label_clusters(df_export_mcplayhd_concept)
# df_cluster_labels_mcplayhd_concept.to_csv(CLUSTER_LABELS_PATHS[METHOD], index=False)

In [ ]:
# temp
df_cluster_labels_mcplayhd_concept.head(10)